We load databricksHelpers and pipelineTests into session memory via %run. We then instantiate DatabricksHelpers for our multi-table Bronze ingestion exercise, clear the scratch workspace directory for session idempotency, and provision the catalog and schemas (bronze, silver, gold).

In [0]:
%run ../../Include/DatabricksHelpers

In [0]:
%run ../../Include/PipelineTests

In [0]:
helpers = DatabricksHelpers(dbutils)

current_user = helpers.current_user()
project_root = helpers.project_root()

print(f"Running pipeline as: {current_user}")
print(f"Project Root:         {project_root}")

In [0]:
catalog_name = "f1_dev"
schemas = ["bronze", "silver", "gold"]

spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
for schema_name in schemas:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

In [0]:
# ASSERTION 1: Verify helpers and test suite functions are present in global memory
assert "DatabricksHelpers" in globals(), "❌ TEST FAILED: 'DatabricksHelpers' class not loaded!"
assert "test_dataframe_not_empty" in globals(), "❌ TEST FAILED: 'test_dataframe_not_empty' function not loaded!"

# ASSERTION 2: Verify active user context and clean exercise directory
assert "@" in current_user, f"❌ TEST FAILED: Invalid user: '{current_user}'"

# ASSERTION 3: Verify all medallion schemas were provisioned
catalog_schemas = [row["databaseName"] for row in spark.sql(f"SHOW SCHEMAS IN {catalog_name}").collect()]
for schema_name in schemas:
    assert schema_name in catalog_schemas, f"❌ TEST FAILED: Schema '{schema_name}' missing from catalog '{catalog_name}'!"

print(f"✓ TEST PASSED: Environment initialized for user '{current_user}'. Catalog '{catalog_name}' and schemas {schemas} are ready.")

Instead of relying on relative working-directory assumptions (data/), we dynamically compute the absolute project_root relative to this notebook's path. We map all 13 source CSV files and verify their existence on disk before initiating Spark reads.

In [0]:
import os

# 1. Discover notebook workspace path and derive project root
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
notebook_dir = os.path.dirname(f"/Workspace{notebook_path}")
project_root = os.path.normpath(os.path.join(notebook_dir, "../../"))
data_dir = os.path.join(project_root, "data")

# 2. Source file map
csv_files = {
    "circuits": "circuits.csv",
    "constructor_standings": "constructor_standings.csv", 
    "constructors": "constructors.csv",
    "driver_standings": "driver_standings.csv",
    "drivers": "drivers.csv",
    "lap_times": "lap_times.csv",
    "pit_stops": "pit_stops.csv",
    "qualifying": "qualifying.csv",
    "races": "races.csv",
    "results": "results.csv",
    "seasons": "seasons.csv",
    "sprint_results": "sprint_results.csv",
    "status": "status.csv",
}

# 3. Construct dictionary of resolved spark paths (`file:/Workspace/...`)
resolved_file_paths = {
    table_name: f"file:{os.path.join(data_dir, file_name)}"
    for table_name, file_name in csv_files.items()
}

In [0]:
# ASSERTION: Check that all 13 CSV files physically exist on local workspace disk
missing_files = []
for table_name, file_name in csv_files.items():
    raw_path = os.path.join(data_dir, file_name)
    if not os.path.exists(raw_path):
        missing_files.append(raw_path)

assert len(missing_files) == 0, f"❌ TEST FAILED: Missing source files on disk:\n" + "\n".join(missing_files)

print(f"✓ TEST PASSED: All {len(csv_files)} source CSV files verified in '{data_dir}'.")

We replace Pandas (pd.read_csv) with native PySpark reader (spark.read.csv) to distribute file reading across cluster workers. We loop through all 13 files, ingest them natively, and persist each as an idempotent Delta table in f1_dev.bronze.

In [0]:
ingested_counts = {}

for table_name, spark_file_path in resolved_file_paths.items():
    target_table = f"{catalog_name}.bronze.{table_name}"
    
    # Read natively using PySpark
    df_temp = (
        spark.read
        .option("header", "true")
        .option("inferSchema", "true")
        .csv(spark_file_path)
    )
    
    # Save as Delta Table
    (
        df_temp.write
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(target_table)
    )
    
    ingested_counts[table_name] = df_temp.count()

In [0]:
# ASSERTION: Reusable test suite validations for each generated Delta table
for table_name, expected_row_count in ingested_counts.items():
    # 1. Table existence and row parity check
    test_table_exists_and_row_count(spark, catalog_name, "bronze", table_name, expected_row_count)
    
    # 2. DataFrame non-empty assertion
    df_table = spark.table(f"{catalog_name}.bronze.{table_name}")
    test_dataframe_not_empty(df_table, f"bronze.{table_name}")

print(f"✓ TEST PASSED: All {len(ingested_counts)} Bronze Delta tables persisted and validated.")

We attach classification and PII governance tags to all 13 Bronze tables in Unity Catalog. We then query the information_schema.table_tags table to ensure metadata compliance.

In [0]:
# Configuration array: (table_name, classification, contains_pii)
tables_config = [
    ("circuits", "public", "false"),
    ("constructor_standings", "public", "false"),
    ("constructors", "public", "false"),
    ("driver_standings", "public", "false"),
    ("drivers", "public", "true"),  # PII: Contains driver names & DOB
    ("lap_times", "public", "false"),
    ("pit_stops", "public", "false"),
    ("qualifying", "public", "false"),
    ("races", "public", "false"),
    ("results", "public", "false"),
    ("seasons", "public", "false"),
    ("sprint_results", "public", "false"),
    ("status", "public", "false")
]

# Apply Governance Tags
for table_name, classification, has_pii in tables_config:
    full_table_path = f"{catalog_name}.bronze.{table_name}"
    spark.sql(f"""
        ALTER TABLE {full_table_path} SET TAGS (
            'classification' = '{classification}',
            'contains_pii' = '{has_pii}'
        )
    """)

In [0]:
# ASSERTION: Verify all governance tags using reusable test function
for table_name, classification, has_pii in tables_config:
    expected_tags = {
        "classification": classification,
        "contains_pii": has_pii
    }
    test_table_tags(spark, catalog_name, "bronze", table_name, expected_tags)

print("🎉 BRONZE MULTI-TABLE INGESTION PIPELINE FULLY EXECUTED & VERIFIED!")